<a href="https://colab.research.google.com/github/madeseor/estructuras-de-bases-de-datos/blob/main/arboles/Arboles-de-Busqueda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ÁRBOLES DE BÚSQUEDA ABB Y B+


**PROBLEMA:** Sistema de Búsqueda de Estudiantes

In [6]:
# Codigo co-creado con IA

import random #generar modulos aleatorios
import time  #calcular tiempos de ejecucion
import statistics #calcular mediana en los tiempos
import sys #aumentar limite de recursion

#   PASO 1: GENERAR n ESTUDIANTES ALEATORIAMENTE

nombres = ["Ana","Carlos","María","Luis","Sofía","Pedro",
           "Laura","Jorge","Valeria","Miguel", "Santiago","Clara","Esteban","Michael","Melany","Genesis","Carlos","Ester"]

# Genera estudiantes hasta tener exactamente el numero que se quiera unico. en este caso 100 mil
estudiantes_dict = {}                   #usamos dicionario porque no permite claves duplicadas
while len(estudiantes_dict) < 100000:   #fijamos el valor de estudiantes que queremos generar
    id = random.randint(1000, 999999)   #el rango debe ser mayor para que se eliminen duplicados y sigamos teniendo la cantidad de estudiantes que queremos
    if id not in estudiantes_dict:
        estudiantes_dict[id] = { #asignaciones
            "id": id,
            "nombre": random.choice(nombres),
            "promedio": round(random.uniform(6.0, 10.0), 1)
        }

estudiantes = list(estudiantes_dict.values())  #convertimos en lista para recorrerlo mas facilmente
estudiantes = list({e["id"]: e for e in estudiantes}.values())

#Segun la manera en que se quieran generar los datos, si aleatoria o en orden.
estudiantes.sort(key=lambda e: e["id"])   # ← IDs EN ORDEN
#random.shuffle(estudiantes)                # ← IDs ALEATORIOS  -- IMPORTANTE

print(f" {len(estudiantes)} estudiantes generados.")
print(f"Primeros 3: {[(e['id'], e['nombre']) for e in estudiantes[:3]]}") #va a mostrar los primeros tres como confirmacion de que se crearon bien

#Busca n aleatorios, si se quieren tener 200, 300 o demas, se modifica aqui y en los prints al final para no
#generar confuciones.
ids_buscar = random.sample([e["id"] for e in estudiantes], 400)

#   FUNCIÓN DE MEDICIÓN DE TIEMOS LIMPIOS:
#   - time.perf_counter: reloj de alta resolución, solo CPU, medicion de tiempos sin ruidos
#   - 15 repeticiones, descarta 20% inferior y 20% superior
#   - Retorna la mediana → elimina ruido del SO, disco, scheduler


def medir_tiempo_limpio(fn_busqueda, ids, repeticiones=15): #Define la función que mide tiempos eliminando el ruido externo. Recibe la función de búsqueda a medir, la lista de IDs a buscar, y cuántas veces repetir (por defecto 15).
    tiempos = []
    for _ in range(repeticiones): #iteraciones
        t0 = time.perf_counter() #perf_counter es un reloj de pythom qie mide en tiempo real desde CPU sin afectarse por cambios del SI
        for id in ids:
            fn_busqueda(id)
        tiempos.append(time.perf_counter() - t0)
    tiempos.sort()
    corte = max(1, int(len(tiempos) * 0.20)) #recorta al menos un valor aunque la lista sea muy pequena
    tiempos_limpios = tiempos[corte:-corte]
    return statistics.median(tiempos_limpios)

#   LISTA — búsqueda lineal (sin árbol)

def buscar_lista(id):
    for e in estudiantes:
        if e["id"] == id:
            return e
    return None

tiempo_lista = medir_tiempo_limpio(buscar_lista, ids_buscar)
print(f"\n Lista     : {tiempo_lista:.6f} segundos (400 búsquedas, mediana limpia)")

#   ABB CON BALANCEO AVL
#   AVL: El arbol se autobalancea despues de cada insercion. Ningun arbol puede tener sus dos ramas
#.  con una diferencia de altura superior a 1 y si es asi, se rota

#Define la estructura de cada nodo del árbol. Guarda los datos del estudiante, las referencias al hijo izquierdo y derecho
#(inicialmente vacíos), y la altura del nodo (inicialmente 1 porque una hoja tiene altura 1).
class Nodo:
    def __init__(self, id, nombre, promedio):
        self.id = id; self.nombre = nombre; self.promedio = promedio
        self.izq = None; self.der = None; self.altura = 1

def altura(n): return n.altura if n else 0
def factor(n): return altura(n.izq) - altura(n.der)

def rotar_der(y):
    x = y.izq; y.izq = x.der; x.der = y
    y.altura = 1 + max(altura(y.izq), altura(y.der))
    x.altura = 1 + max(altura(x.izq), altura(x.der))
    return x

def rotar_izq(x):
    y = x.der; x.der = y.izq; y.izq = x
    x.altura = 1 + max(altura(x.izq), altura(x.der))
    y.altura = 1 + max(altura(y.izq), altura(y.der))
    return y

def balancear(nodo):
    nodo.altura = 1 + max(altura(nodo.izq), altura(nodo.der))
    fb = factor(nodo)
    if fb > 1:
        if factor(nodo.izq) < 0: nodo.izq = rotar_izq(nodo.izq)
        return rotar_der(nodo)
    if fb < -1:
        if factor(nodo.der) > 0: nodo.der = rotar_der(nodo.der)
        return rotar_izq(nodo)
    return nodo

def insertar_abb(nodo, id, nombre, promedio):
    if not nodo: return Nodo(id, nombre, promedio)
    if id < nodo.id:   nodo.izq = insertar_abb(nodo.izq, id, nombre, promedio)
    elif id > nodo.id: nodo.der = insertar_abb(nodo.der, id, nombre, promedio)
    return balancear(nodo)

def buscar_abb(nodo, id):
    if not nodo: return None
    if id == nodo.id: return nodo
    return buscar_abb(nodo.izq, id) if id < nodo.id else buscar_abb(nodo.der, id)

def listar_abb(nodo, lista=[]):
    if nodo:
        listar_abb(nodo.izq, lista)
        lista.append((nodo.id, nodo.nombre, nodo.promedio))
        listar_abb(nodo.der, lista)
    return lista

sys.setrecursionlimit(20000)

raiz_abb = None
for e in estudiantes:
    raiz_abb = insertar_abb(raiz_abb, e["id"], e["nombre"], e["promedio"])

tiempo_abb = medir_tiempo_limpio(lambda id: buscar_abb(raiz_abb, id), ids_buscar)
print(f" ABB (AVL) : {tiempo_abb:.6f} segundos (400 búsquedas, mediana limpia)")

#
#   ÁRBOL B

T = 50

class NodoB:
    def __init__(self, hoja=True):
        self.hoja   = hoja
        self.claves = []
        self.hijos  = []

class ArbolB:
    def __init__(self):
        self.raiz = NodoB(hoja=True)

    def buscar(self, id):
        return self._buscar(self.raiz, id)

    def _buscar(self, nodo, id):
        i = 0
        while i < len(nodo.claves) and id > nodo.claves[i][0]:
            i += 1
        if i < len(nodo.claves) and nodo.claves[i][0] == id:
            return nodo.claves[i][1]
        if nodo.hoja:
            return None
        return self._buscar(nodo.hijos[i], id)

    def insertar(self, id, datos):
        raiz = self.raiz
        if len(raiz.claves) == 2 * T - 1:
            nueva_raiz = NodoB(hoja=False)
            nueva_raiz.hijos.append(self.raiz)
            self._dividir_hijo(nueva_raiz, 0)
            self.raiz = nueva_raiz
        self._insertar_no_lleno(self.raiz, id, datos)

    def _insertar_no_lleno(self, nodo, id, datos):
        i = len(nodo.claves) - 1
        if nodo.hoja:
            nodo.claves.append(None)
            while i >= 0 and id < nodo.claves[i][0]:
                nodo.claves[i + 1] = nodo.claves[i]
                i -= 1
            nodo.claves[i + 1] = (id, datos)
        else:
            while i >= 0 and id < nodo.claves[i][0]:
                i -= 1
            i += 1
            if len(nodo.hijos[i].claves) == 2 * T - 1:
                self._dividir_hijo(nodo, i)
                if id > nodo.claves[i][0]:
                    i += 1
            self._insertar_no_lleno(nodo.hijos[i], id, datos)

    def _dividir_hijo(self, padre, i):
        t   = T
        y   = padre.hijos[i]
        z   = NodoB(hoja=y.hoja)
        mid = t - 1
        padre.claves.insert(i, y.claves[mid])
        z.claves = y.claves[mid + 1:]
        y.claves = y.claves[:mid]
        if not y.hoja:
            z.hijos = y.hijos[t:]
            y.hijos = y.hijos[:t]
        padre.hijos.insert(i + 1, z)

    def listar(self):
        resultado = []
        self._listar(self.raiz, resultado)
        return resultado

    def _listar(self, nodo, resultado):
        for i, clave in enumerate(nodo.claves):
            if not nodo.hoja:
                self._listar(nodo.hijos[i], resultado)
            resultado.append(clave)
        if not nodo.hoja:
            self._listar(nodo.hijos[len(nodo.claves)], resultado)

arbol_b = ArbolB()
for e in estudiantes:
    arbol_b.insertar(e["id"], e)

tiempo_b = medir_tiempo_limpio(lambda id: arbol_b.buscar(id), ids_buscar)
print(f" B         : {tiempo_b:.6f} segundos (400 búsquedas, mediana limpia)")


#   ÁRBOL B+

ORDEN = 50

class NodoBP:
    def __init__(self, hoja=False):
        self.hoja = hoja
        self.claves = []; self.hijos = []
        self.siguiente = None

class ArbolBPlus:
    def __init__(self): self.raiz = NodoBP(hoja=True)

    def buscar(self, id):
        nodo = self.raiz
        while not nodo.hoja:
            i = 0
            while i < len(nodo.claves) and id >= nodo.claves[i]: i += 1
            nodo = nodo.hijos[i]
        for i, c in enumerate(nodo.claves):
            if c == id: return nodo.hijos[i]
        return None

    def insertar(self, id, datos):
        res = self._ins(self.raiz, id, datos)
        if res:
            nueva = NodoBP()
            nueva.claves = [res[0]]; nueva.hijos = [self.raiz, res[1]]
            self.raiz = nueva

    def _ins(self, nodo, id, datos):
        if nodo.hoja:
            i = 0
            while i < len(nodo.claves) and id > nodo.claves[i]: i += 1
            nodo.claves.insert(i, id); nodo.hijos.insert(i, datos)
            if len(nodo.claves) >= ORDEN: return self._div_hoja(nodo)
        else:
            i = 0
            while i < len(nodo.claves) and id >= nodo.claves[i]: i += 1
            res = self._ins(nodo.hijos[i], id, datos)
            if res:
                nodo.claves.insert(i, res[0]); nodo.hijos.insert(i+1, res[1])
                if len(nodo.claves) >= ORDEN: return self._div_int(nodo)
        return None

    def _div_hoja(self, nodo):
        mid = len(nodo.claves)//2
        nuevo = NodoBP(hoja=True)
        nuevo.claves = nodo.claves[mid:]; nuevo.hijos = nodo.hijos[mid:]
        nodo.claves  = nodo.claves[:mid]; nodo.hijos  = nodo.hijos[:mid]
        nuevo.siguiente = nodo.siguiente; nodo.siguiente = nuevo
        return (nuevo.claves[0], nuevo)

    def _div_int(self, nodo):
        mid = len(nodo.claves)//2
        nuevo = NodoBP()
        nuevo.claves = nodo.claves[mid+1:]; nuevo.hijos = nodo.hijos[mid+1:]
        c = nodo.claves[mid]
        nodo.claves = nodo.claves[:mid];    nodo.hijos  = nodo.hijos[:mid+1]
        return (c, nuevo)

    def listar(self):
        nodo = self.raiz
        while not nodo.hoja: nodo = nodo.hijos[0]
        res = []
        while nodo:
            for i, c in enumerate(nodo.claves): res.append((c, nodo.hijos[i]))
            nodo = nodo.siguiente
        return res

arbol_bp = ArbolBPlus()
for e in estudiantes:
    arbol_bp.insertar(e["id"], e)

tiempo_bplus = medir_tiempo_limpio(lambda id: arbol_bp.buscar(id), ids_buscar)
print(f" B+        : {tiempo_bplus:.6f} segundos (400 búsquedas, mediana limpia)")


#   RESULTADOS FINALES

tiempos = {"ABB (AVL)": tiempo_abb, "B": tiempo_b, "B+": tiempo_bplus}
ganador = min(tiempos, key=tiempos.get)

print("\n" + "="*55)
print("   Resultados para", len(estudiantes), "estudiantes:")
print("   (tiempos mediana limpia — sin ruido del SO)")
print("="*55)
print(f"  Lista  : {tiempo_lista:.6f} segundos")
print(f"  ABB    : {tiempo_abb:.6f} segundos  (¡{tiempo_lista/tiempo_abb:.0f}x más rápido que lista!)")
print(f"  B      : {tiempo_b:.6f} segundos  (¡{tiempo_lista/tiempo_b:.0f}x más rápido que lista!)")
print(f"  B+     : {tiempo_bplus:.6f} segundos  (¡{tiempo_lista/tiempo_bplus:.0f}x más rápido que lista!)")
print("="*55)
print(f"   El más rápido fue: {ganador}")
print("="*55)


#   LISTAR primeros 5 en orden por ID (los tres árboles)

print("\n Primeros 5 en orden — ABB:")
for f in listar_abb(raiz_abb, [])[:5]:
    print(f"  ID: {f[0]} | Nombre: {f[1]} | Promedio: {f[2]}")

print("\n Primeros 5 en orden — B:")
for clave, datos in arbol_b.listar()[:5]:
    print(f"  ID: {clave} | Nombre: {datos['nombre']} | Promedio: {datos['promedio']}")

print("\n Primeros 5 en orden — B+:")
for c, d in arbol_bp.listar()[:5]:
    print(f"  ID: {c} | Nombre: {d['nombre']} | Promedio: {d['promedio']}")

 100000 estudiantes generados.
Primeros 3: [(1002, 'Jorge'), (1014, 'Valeria'), (1016, 'Santiago')]

 Lista     : 6.845152 segundos (400 búsquedas, mediana limpia)
 ABB (AVL) : 0.001328 segundos (400 búsquedas, mediana limpia)
 B         : 0.006977 segundos (400 búsquedas, mediana limpia)
 B+        : 0.001635 segundos (400 búsquedas, mediana limpia)

   Resultados para 100000 estudiantes:
   (tiempos mediana limpia — sin ruido del SO)
  Lista  : 6.845152 segundos
  ABB    : 0.001328 segundos  (¡5154x más rápido que lista!)
  B      : 0.006977 segundos  (¡981x más rápido que lista!)
  B+     : 0.001635 segundos  (¡4186x más rápido que lista!)
   El más rápido fue: ABB (AVL)

 Primeros 5 en orden — ABB:
  ID: 1002 | Nombre: Jorge | Promedio: 7.2
  ID: 1014 | Nombre: Valeria | Promedio: 8.5
  ID: 1016 | Nombre: Santiago | Promedio: 9.5
  ID: 1021 | Nombre: Ana | Promedio: 7.3
  ID: 1030 | Nombre: María | Promedio: 7.9

 Primeros 5 en orden — B:
  ID: 1002 | Nombre: Jorge | Promedio: 7.2
